# OCR a PDF with Docling, Chunk It, and Store in pgvector

This notebook is the **ingestion** step of a RAG pipeline. It takes the PDF downloaded in the previous notebook and turns it into searchable chunks:

```
sample.pdf --> [ Docling: OCR + parse ] --> text
           --> [ chunk into passages ]   --> chunks
           --> [ embed each chunk ]      --> vectors  --> [ store in pgvector ]
           --> [ index the chunk text ]              --> [ BM25 via pg_textsearch ]
```

1. **Docling** reads the PDF, running **OCR** on any scanned/image pages and parsing the layout into clean text.
2. We **chunk** the text into passages small enough to embed and retrieve well.
3. We **embed** each chunk into a vector.
4. We **store** the chunks and their vectors in the **pgvector** database from the `1_install_pgvector` step, ready for retrieval.
5. We also index the same chunk text for **BM25 keyword search** (`pg_textsearch`) - used later in the **hybrid RAG** section.

Prerequisite: run `3_download_from_s3.ipynb` first so `sample.pdf` exists locally, and make sure your database is running with the `vector` and `pg_textsearch` extensions (see `1_install_pgvector`).

## Dependencies

This step is heavier than the upload/download notebooks: `docling` (OCR + PDF parsing), `sentence-transformers` + `langchain-huggingface` (local embeddings), and `langchain-postgres` + `pgvector` + `psycopg` (the vector store). All are listed in `../requirements.txt`:

```bash
pip install -r requirements.txt
```

A fresh virtual environment is recommended. On first run, Docling downloads its OCR/layout models and the embedding model downloads too, so the first execution takes a while.

## Configuration

Add the pgvector connection details to the same `.env` file used by the other notebooks. These defaults match the `1_install_pgvector` manifests; set `PG_PASSWORD` to the password you put in `01-secret.yaml`.

This notebook runs **inside the same Kubernetes cluster**, so `PG_HOST` is the service DNS name `pgvector` (reachable as `pgvector` in the same `default` namespace, or `pgvector.default.svc.cluster.local` from another namespace) - no port-forward needed.

```dotenv
PG_HOST=pgvector       # in-cluster service name (see 05-service.yaml)
PG_PORT=5432
PG_USER=raguser
PG_PASSWORD=change-me-please
PG_DB=ragdb
```

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv("var.env")

PG_HOST = os.environ.get("PG_HOST", "pgvector")   # in-cluster service DNS name
PG_PORT = os.environ.get("PG_PORT", "5432")
PG_USER = os.environ.get("PG_USER", "raguser")
PG_PASSWORD = os.environ.get("PG_PASSWORD", "change-me-please")
PG_DB = os.environ.get("PG_DB", "ragdb")

# SQLAlchemy-style URL used by langchain-postgres (note the +psycopg driver).
CONNECTION = f"postgresql+psycopg://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"
print("connecting to:", f"postgresql+psycopg://{PG_USER}:***@{PG_HOST}:{PG_PORT}/{PG_DB}")

pdf_path = Path("sample.pdf")
if not pdf_path.exists():
    raise FileNotFoundError(
        "sample.pdf not found. Run 3_download_from_s3.ipynb first to download it from Deka Box."
    )
print("PDF to ingest:", pdf_path.resolve())

## Step 1: OCR and parse the PDF with Docling

Docling converts the PDF into a structured document. We enable **`do_ocr`** so any scanned or image-based pages are run through OCR; born-digital pages use their existing text layer automatically. `do_table_structure` recovers tables as structured content.

The first run downloads Docling's models, so it can take a minute or two.

In [ ]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = True              # OCR scanned/image pages
pipeline_options.do_table_structure = True  # recover table structure
# To force OCR on every page even when a text layer exists:
# pipeline_options.ocr_options.force_full_page_ocr = True

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

result = converter.convert(str(pdf_path))
doc = result.document

markdown = doc.export_to_markdown()
print(f"parsed document, {len(markdown)} characters of text\n")
print(markdown[:800])

## Step 2: Chunk the text

A whole document is too big to embed as one vector and too coarse to retrieve precisely. Docling's **`HybridChunker`** splits the parsed document into passages along its structure (headings, sections) while keeping each chunk within a sensible token budget. `contextualize` prepends the relevant heading path to each chunk so it stays meaningful on its own.

We wrap each chunk in a LangChain `Document` with a bit of metadata so we can trace results back to the source.

In [ ]:
from docling.chunking import HybridChunker
from langchain_core.documents import Document

chunker = HybridChunker()

documents = []
for i, chunk in enumerate(chunker.chunk(dl_doc=doc)):
    text = chunker.contextualize(chunk=chunk)   # chunk text enriched with its headings
    documents.append(Document(
        page_content=text,
        metadata={"source": pdf_path.name, "chunk_index": i},
    ))

print(f"created {len(documents)} chunks")
if documents:
    print("\n--- first chunk ---")
    print(documents[0].page_content[:500])

## Step 3: Build the embedding model

Each chunk must become a vector. We use a small, local **sentence-transformers** model (`all-MiniLM-L6-v2`, 384 dimensions) via `langchain-huggingface`. It runs on CPU and downloads once.

The *same* embedding model must be used later when querying, so the query vector lives in the same space as the stored ones.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# quick sanity check on the vector size
sample_vec = embeddings.embed_query("hello")
print("embedding dimensions:", len(sample_vec))

## Step 4: Store the chunks in pgvector

`PGVector` from `langchain-postgres` manages the table for us: it creates a **collection** (a logical group of documents) and, on `add_documents`, embeds each chunk and inserts the text, metadata, and vector. `use_jsonb=True` stores metadata as JSONB so it can be filtered later.

This relies on the `vector` extension, which the `1_install_pgvector` step already enabled in the database.

In [ ]:
from langchain_postgres import PGVector

COLLECTION_NAME = "rag_documents"

vector_store = PGVector(
    embeddings=embeddings,
    collection_name=COLLECTION_NAME,
    connection=CONNECTION,
    use_jsonb=True,
)

ids = vector_store.add_documents(documents)
print(f"stored {len(ids)} chunks in collection '{COLLECTION_NAME}'")

## Step 5: Also index the chunks for BM25 (pg_textsearch)

Dense vector search is great for meaning, but hybrid RAG also needs **keyword** search. We make the same chunks BM25-searchable by adding a `pg_textsearch` **BM25 index** over the text `PGVector` just stored - no second copy of the data.

> **Note:** this index (`bm25_chunks_idx`) is used later in the **hybrid RAG** section (`2_hybrid_rag`), where BM25 keyword results are fused with pgvector similarity results.

In [ ]:
import psycopg

# Open a raw psycopg connection (langchain-postgres used the SQLAlchemy URL; the
# BM25 index is plain SQL, so we connect directly).
with psycopg.connect(
    f"host={PG_HOST} port={PG_PORT} dbname={PG_DB} user={PG_USER} password={PG_PASSWORD}"
) as pg_conn:
    pg_conn.autocommit = True
    with pg_conn.cursor() as cur:
        # pg_textsearch must be installed and preloaded (see 1_install_pgvector).
        cur.execute("SELECT 1 FROM pg_extension WHERE extname = 'pg_textsearch';")
        if cur.fetchone() is None:
            raise RuntimeError("pg_textsearch is not installed. See 1_install_pgvector.")

        # PGVector stored each chunk's text in the `document` column of its
        # `langchain_pg_embedding` table. A BM25 index over that column makes the
        # SAME chunks searchable by keyword - no need to store them twice.
        # Created once; safe to re-run (IF NOT EXISTS).
        cur.execute(
            "CREATE INDEX IF NOT EXISTS bm25_chunks_idx ON langchain_pg_embedding "
            "USING bm25(document) WITH (text_config='english');"
        )

print("BM25 index ready over the stored chunks")
print("NOTE: the hybrid RAG section reuses this index for keyword search.")

## Step 6: Verify with a similarity search

To confirm the data is stored and searchable, we run a vector similarity search. The query is embedded with the same model, and pgvector returns the nearest chunks - this is exactly the retrieval step the RAG chain will use later.

In [ ]:
query = "What does this document say?"
results = vector_store.similarity_search(query, k=3)

print(f"top {len(results)} chunks for: {query!r}\n")
for rank, doc_hit in enumerate(results, start=1):
    print(f"[{rank}] source={doc_hit.metadata.get('source')} chunk={doc_hit.metadata.get('chunk_index')}")
    print("    ", doc_hit.page_content[:200].replace("\n", " "))
    print()

## Recap

- **Docling** turns a PDF into clean text, running **OCR** on scanned pages (`do_ocr=True`) and parsing layout/tables - no separate OCR engine wiring needed.
- **`HybridChunker`** splits the parsed document into structure-aware passages; `contextualize` keeps each chunk self-contained.
- Each chunk is **embedded** with a local `sentence-transformers` model - use the *same* model at query time.
- **`PGVector`** stores the chunks, metadata, and vectors in the pgvector database and makes them searchable with `similarity_search`.
- We also added a **BM25 index** (`pg_textsearch`) over the same chunk text so they can be searched by keyword - this is reused in the **hybrid RAG** section.

The document is now indexed for both **dense** (vector) and **sparse** (BM25) retrieval. The next step of a RAG pipeline retrieves these chunks for a question and feeds them to an LLM to generate a grounded answer.